# Lecture des émotions individuelles par sondes

Les agents observent chaque pair séparément. Comment un réseau entraîné intègre-t-il
les cinq signaux : comme une moyenne, ou en suivant le pair qui s'écarte ?
La lecture des poids ne répond pas à la question, on sonde donc les réseaux
(EXPERIMENT_TRACKER, décidé le 14/10/2026 ; méthode journal, *Reading of individual
affective signals*). Aucune simulation n'est modifiée.

1. rejouer des épisodes greedy avec les poids finaux, pour collecter les observations
   que la politique visite réellement ;
2. remplacer les 5 cases sociales par des vecteurs où un pair, à position aléatoire,
   s'écarte des 4 autres ;
3. mesurer ΔQ = Q(o, exploiter) − Q(o, s'abstenir) ;
4. régresser ΔQ sur la moyenne et sur la valeur déviante, globalement et par position ;
5. comparer les conditions. L'unité d'analyse est le run.

Seules les conditions où les émotions sont perçues ont un sens ici (C, D, puis les
niveaux des axes 2 et 3). Le batch `beta3` sert de valeur par défaut : à remplacer par
les runs finaux.

In [ ]:
import glob
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.stats import mannwhitneyu

REPO = Path("..").resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import main_gt_ais as M

RUN_DIR = REPO / "csv_results" / "beta3" / "global_v1"   # or csv_results/runs/<config_id>
CONDITIONS = ("C", "D")     # first letter of the condition label
EPISODES_PER_RUN = 2        # greedy episodes replayed to collect observations
OBS_PER_RUN = 400           # visited observations kept per run
PROBES_PER_OBS = 5          # controlled vectors per observation
SEED = 0

## 1. Observations visitées par la politique finale

`save_weights` n'enregistre pas les observations : on rejoue la politique avec la
configuration du run, lue dans son JSON. Chaque run recharge les poids de ses 6 agents.

In [ ]:
def load_run(weights_path):
    """environment and agents of one run, with its final weights and epsilon = 0."""
    weights = torch.load(weights_path, weights_only=True)
    with open(str(weights_path).replace("_weights.pt", "_config.json")) as f:
        config = json.load(f)
    assert config["emotion_type"] == "vector", "the probes need individual emotion slots"
    M.REWARD_SCHEME = config["reward_scheme"]
    M.PARAMS_DQN = config["params_dqn"]
    env, agents = M.init_env(weights["condition"])
    for agent, state in zip(agents, weights["final"]):
        agent.policy_network.load_state_dict(state)
        agent.epsilon = 0.0
    return env, agents, weights


def collect_observations(env, agents, episodes, rng, keep):
    """(agent index, observation) pairs visited by the greedy policies."""
    visited = []
    for _ in range(episodes):
        obs = env.reset()
        for _ in range(M.MAX_STEPS):
            visited += [(i, o.copy()) for i, o in enumerate(obs)]
            obs, _, done, _ = env.step([a.select_action(obs[i]) for i, a in enumerate(agents)])
            if done:
                break
    picks = rng.choice(len(visited), size=min(keep, len(visited)), replace=False)
    return [visited[k] for k in picks]

## 2–3. Sondes et réponse ΔQ

Dans chaque vecteur, 4 pairs partagent une valeur de base et le 5ᵉ, à position
aléatoire, prend une valeur déviante. Les deux sont tirées indépendamment sur l'échelle
des émotions : la moyenne et la valeur déviante varient donc en grande partie
indépendamment (corrélation théorique ≈ 0,24, vérifiée plus bas).

In [ ]:
def probe_run(weights_path, rng):
    env, agents, weights = load_run(weights_path)
    low, high = env.social_calculator.emotion_range
    rows = []
    for i, base_obs in collect_observations(env, agents, EPISODES_PER_RUN, rng, OBS_PER_RUN):
        n = PROBES_PER_OBS
        base = rng.uniform(low, high, n)
        deviant = rng.uniform(low, high, n)
        position = rng.integers(0, 5, n)
        probes = np.tile(base_obs, (n, 1))
        probes[:, 2:7] = base[:, None]
        probes[np.arange(n), 2 + position] = deviant
        with torch.no_grad():
            q = agents[i].combined_q(agents[i].policy_network(torch.tensor(probes, dtype=torch.float32)))
        dq = (q[:, 1] - q[:, 0]).numpy()
        rows.append(pd.DataFrame({"agent": i, "base": base, "deviant": deviant,
                                  "position": position, "dq": dq}))
    probes = pd.concat(rows, ignore_index=True)
    probes["mean"] = (4 * probes["base"] + probes["deviant"]) / 5
    return probes, weights["condition"]["label"][0]

## 4. Régression par run

ΔQ ~ moyenne + valeur déviante. Une lecture « moyenne » donne un coefficient nul à la
valeur déviante. Un coefficient négatif sur la moyenne signifie que l'agent exploite
moins quand les autres vont mieux. La pente de la valeur déviante est aussi estimée
séparément selon que le pair est plus mal loti que les autres (`slope_below`) ou mieux
loti (`slope_above`), et par position, pour voir si certains pairs comptent plus.

In [ ]:
def slopes(d, columns):
    X = np.column_stack([np.ones(len(d))] + [d[c] for c in columns])
    return np.linalg.lstsq(X, d["dq"].to_numpy(), rcond=None)[0][1:]


def run_indices(probes):
    coef_mean, coef_dev = slopes(probes, ["mean", "deviant"])
    below = probes[probes.deviant < probes.base]
    above = probes[probes.deviant > probes.base]
    out = {"coef_mean": coef_mean, "coef_deviant": coef_dev,
           "slope_below": slopes(below, ["base", "deviant"])[1],
           "slope_above": slopes(above, ["base", "deviant"])[1],
           "corr_mean_deviant": probes["mean"].corr(probes["deviant"]),
           "dq_sd": probes["dq"].std()}
    for k in range(5):
        out[f"position_{k}"] = slopes(probes[probes.position == k], ["base", "deviant"])[1]
    return out


rng = np.random.default_rng(SEED)
M.set_global_seed(SEED)
files = sorted(p for p in Path(RUN_DIR).glob("*_weights.pt"))
results = []
for path in files:
    if torch.load(path, weights_only=True)["condition"]["label"][0] not in CONDITIONS:
        continue
    probes, condition = probe_run(path, rng)
    results.append({"run": path.name.replace("_weights.pt", ""), "condition": condition,
                    **run_indices(probes)})
results = pd.DataFrame(results)
print(f"{len(results)} runs sondés sur {len(files)} fichiers de poids")
results.groupby("condition").size()

## 5. Comparaison des conditions

La question de départ : la motivation empathique oriente-t-elle l'attention vers le pair
le plus en manque ? Si oui, `slope_below` doit être plus forte en valeur absolue en D
qu'en C, et plus forte que `slope_above` en D. Avec peu de runs par condition, ces
valeurs restent indicatives.

In [ ]:
indices = ["coef_mean", "coef_deviant", "slope_below", "slope_above", "corr_mean_deviant", "dq_sd"]
summary = results.groupby("condition")[indices].agg(["mean", "std"]).round(3).T
display(summary)

rows = []
if set(results.condition) >= {"C", "D"}:
    for name in indices[:4]:
        c = results.loc[results.condition == "C", name]
        d = results.loc[results.condition == "D", name]
        u, p = mannwhitneyu(c, d, alternative="two-sided")
        rows.append({"indice": name, "C": c.mean(), "D": d.mean(), "U": u, "p": p})
    display(pd.DataFrame(rows).round(3))

## 6. Certains pairs comptent-ils plus ?

Pente de la valeur déviante selon la position du pair, puis amplitude de ces écarts
entre positions, run par run : les 6 agents d'un run voient les pairs dans un ordre fixe,
donc une position privilégiée durable peut émerger.

In [ ]:
positions = [f"position_{k}" for k in range(5)]
display(results.groupby("condition")[positions].mean().round(3))
results["position_range"] = results[positions].max(axis=1) - results[positions].min(axis=1)
display(results.groupby("condition")["position_range"].agg(["mean", "std"]).round(3))